## ***A Data-Driven Analysis of Tourist Satisfaction in SriLankan Destinations Using Online Review Analytics***

# Import Libraries

In [23]:
# ==
# Core Libraries
# ==
import pandas as pd
import numpy as np

# ==
# Visualization
# ==
import matplotlib.pyplot as plt
import seaborn as sns

# ==
# Text Processing
# ==
import random
import re
import string

# ==
# Natural Language Processing
# ==
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# ==
# Feature Extraction
# ==
from sklearn.feature_extraction.text import CountVectorizer

# ==
# Sentiment Analysis
# ==
from nltk.sentiment import SentimentIntensityAnalyzer

# ==
# Topic Modeling
# ==
from gensim import corpora
from gensim.models import LdaModel

# ==
# Word Cloud
# ==
from wordcloud import WordCloud

# ==
# Display Settings
# ==
pd.set_option('display.max_columns', None)
sns.set_style("whitegrid")

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('vader_lexicon')
nltk.download('punkt_tab') # Added to resolve the LookupError

# Reproducibility
random.seed(42)
np.random.seed(42)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\abaiy\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\abaiy\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\abaiy\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\abaiy\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


# Basic Data Inspection and Pre processing

In [24]:
df = pd.read_csv("Reviews.csv", encoding='latin1')
df.head()

,Location_Name,Located_City,Location,Location_Type,User_ID,User_Location,User_Locale,User_Contributions,Travel_Date,Published_Date,Rating,Helpful_Votes,Title,Text
0,Arugam Bay,Arugam Bay,"Arugam Bay, Eastern Province",Beaches,User 1,"Dunsborough, Australia",en_US,8,2019-07,2019-07-31T07:53:21-04:00,5,1,Best nail spa in Arugam bay on the water!,I had a manicure here and it really was profes...
1,Arugam Bay,Arugam Bay,"Arugam Bay, Eastern Province",Beaches,User 2,"Bendigo, Australia",en_US,4,2019-06,2019-07-21T21:50:11-04:00,4,0,Best for surfing,"Overall, it is a wonderful experience. We visi..."
2,Arugam Bay,Arugam Bay,"Arugam Bay, Eastern Province",Beaches,User 3,"Melbourne, Australia",en_US,13,2019-07,2019-07-15T18:52:55-04:00,5,0,We Love Arugam Bay,"Great place to chill, swim, surf, eat, shop, h..."
3,Arugam Bay,Arugam Bay,"Arugam Bay, Eastern Province",Beaches,User 4,"Ericeira, Portugal",en_US,4,2019-06,2019-07-03T10:32:41-04:00,5,0,Sun and waves.,Good place for surf and a few stores to going ...
4,Arugam Bay,Arugam Bay,"Arugam Bay, Eastern Province",Beaches,User 5,"Pistoia, Italy",en_US,14,2019-07,2019-07-02T17:07:02-04:00,5,0,"Great swimming, surfing, great fish aznd frien...",This place is great for surfing but even if yo...


In [25]:
df.shape

(16156, 14)

In [26]:
df.columns

Index(['Location_Name', 'Located_City', 'Location', 'Location_Type', 'User_ID',
       'User_Location', 'User_Locale', 'User_Contributions', 'Travel_Date',
       'Published_Date', 'Rating', 'Helpful_Votes', 'Title', 'Text'],
      dtype='str')

In [27]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 16156 entries, 0 to 16155
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   Location_Name       16156 non-null  str  
 1   Located_City        16156 non-null  str  
 2   Location            16156 non-null  str  
 3   Location_Type       16156 non-null  str  
 4   User_ID             16156 non-null  str  
 5   User_Location       16156 non-null  str  
 6   User_Locale         16156 non-null  str  
 7   User_Contributions  16156 non-null  int64
 8   Travel_Date         16156 non-null  str  
 9   Published_Date      16156 non-null  str  
 10  Rating              16156 non-null  int64
 11  Helpful_Votes       16156 non-null  int64
 12  Title               16156 non-null  str  
 13  Text                16156 non-null  str  
dtypes: int64(3), str(11)
memory usage: 1.7 MB


In [28]:
df.describe()

,User_Contributions,Rating,Helpful_Votes
count,16156.000000,16156.000000,16156.000000
mean,191.624845,4.167492,0.709458
std,500.100421,1.006840,3.672513
min,1.000000,1.000000,0.000000
25%,18.000000,4.000000,0.000000
50%,54.000000,4.000000,0.000000
75%,155.000000,5.000000,1.000000
max,9010.000000,5.000000,233.000000


In [29]:
df.isnull().sum()

Location_Name         0
Located_City          0
Location              0
Location_Type         0
User_ID               0
User_Location         0
User_Locale           0
User_Contributions    0
Travel_Date           0
Published_Date        0
Rating                0
Helpful_Votes         0
Title                 0
Text                  0
dtype: int64

In [30]:
df.duplicated().sum()

np.int64(0)

In [31]:
df["Rating"].value_counts()

Rating
5    7649
4    5196
3    2166
2     658
1     487
Name: count, dtype: int64

In [32]:
df["Location_Type"].value_counts()

Location_Type
Religious Sites            3017
Beaches                    2110
Farms                      1884
Nature & Wildlife Areas    1557
Museums                    1525
Historic Sites             1519
Gardens                    1354
National Parks             1205
Waterfalls                  933
Bodies of Water             839
Zoological Gardens          213
Name: count, dtype: int64

In [33]:
df["Located_City"].value_counts().head(10)

Located_City
Nuwara Eliya    2221
Anuradhapura    1758
Kandy           1480
Colombo         1171
Sigiriya         763
Habarana         754
Hikkaduwa        515
Galle            511
Jaffna           475
Ella             471
Name: count, dtype: int64

In [34]:
date_columns = ["Travel_Date", "Published_Date"]

for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors="coerce", utc=True)

df.dtypes

Location_Name                         str
Located_City                          str
Location                              str
Location_Type                         str
User_ID                               str
User_Location                         str
User_Locale                           str
User_Contributions                  int64
Travel_Date           datetime64[us, UTC]
Published_Date        datetime64[us, UTC]
Rating                              int64
Helpful_Votes                       int64
Title                                 str
Text                                  str
dtype: object

In [35]:
df = df.drop(columns=["User_ID"])

In [36]:
df["review_length"] = df["Text"].apply(len)          # Creating new features
df["word_count"] = df["Text"].apply(lambda x: len(str(x).split()))
df.info()
df.describe()

<class 'pandas.DataFrame'>
RangeIndex: 16156 entries, 0 to 16155
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype              
---  ------              --------------  -----              
 0   Location_Name       16156 non-null  str                
 1   Located_City        16156 non-null  str                
 2   Location            16156 non-null  str                
 3   Location_Type       16156 non-null  str                
 4   User_Location       16156 non-null  str                
 5   User_Locale         16156 non-null  str                
 6   User_Contributions  16156 non-null  int64              
 7   Travel_Date         16156 non-null  datetime64[us, UTC]
 8   Published_Date      16156 non-null  datetime64[us, UTC]
 9   Rating              16156 non-null  int64              
 10  Helpful_Votes       16156 non-null  int64              
 11  Title               16156 non-null  str                
 12  Text                16156 non-null  str    

,User_Contributions,Rating,Helpful_Votes,review_length,word_count
count,16156.000000,16156.000000,16156.000000,16156.000000,16156.000000
mean,191.624845,4.167492,0.709458,380.618779,70.068705
std,500.100421,1.006840,3.672513,329.177103,60.678855
min,1.000000,1.000000,0.000000,50.000000,1.000000
25%,18.000000,4.000000,0.000000,176.000000,32.000000
50%,54.000000,4.000000,0.000000,282.000000,52.000000
75%,155.000000,5.000000,1.000000,466.000000,86.000000
max,9010.000000,5.000000,233.000000,9430.000000,1700.000000


In [37]:
for value in df["Location_Name"].unique():
    print(value)

Arugam Bay
Bentota Beach
Hikkaduwa Beach
Jungle Beach
Kalutara Beach
Marble Beach
Mirissa Beach
Mount Lavinia Beach
Negombo Beach
Nilaveli Beach
Passikudah Beach
Gregory Lake
Kandy Lake
Tissa Wewa
Twin Baths (Kuttam Pokuna)
Ambewela Farms
Bluefield Tea Gardens
Dambatenne Tea Factory
Damro Labookellie Tea Centre and Tea Garden
Glenloch Tea Factory
Halpewatte Tea Factory Tour
Handunugoda Tea Estate
Pedro Tea Factory
Brief Garden - Bevis Bawa
Hakgala Botanic Gardens
New Ranweli Spice Garden
Royal Botanical Gardens
Victoria Park of Nuwara Eliya
Galle Fort
Jaffna Fort
Liptons Seat
Polonnaruwa
Ritigala Forest Monastery
Sigiriya The Ancient Rock Fortress
Ariyapala Mask Museum
Ceylon Tea Museum
Colombo National Museum
Community Tsunami Museum
Martin Wickramasinghe Folk Museum Complex
Sigiriya Museum
World Buddhist Museum
Bundala National Park
Horton Plains National Park
Kaudulla National Park
Kumana National Park
Minneriya National Park
Pigeon Island National Park
Udawalawe National Park
Kala

In [38]:
for value in df["Located_City"].unique():
    print(value)

Arugam Bay
Bentota
Hikkaduwa
Unawatuna
Kalutara
Trincomalee
Mirissa
Colombo
Negombo
Nilaveli
Kalkudah
Nuwara Eliya
Kandy
Tissamaharama
Anuradhapura
Haputale
Katukitula
Ella
Ahangama
Beruwala
Peradeniya
Galle
Jaffna
Polonnaruwa
Sigiriya
Ambalangoda
Weligatta
Habarana
Ampara
Embilipitiya
Kalametiya
Pinnawala
Deniyaya
Saliyapura
Koslanda
Pussellawa


In [39]:
# Shared location-engineering helpers (define once before validation/extraction cells)
import re

provinces = [
    "Western Province",
    "Central Province",
    "Southern Province",
    "Northern Province",
    "Eastern Province",
    "North Western Province",
    "North Central Province",
    "Uva Province",
    "Sabaragamuwa Province",
]

city_to_district = {
    "Arugam Bay": "Ampara",
    "Colombo": "Colombo",
    "Kandy": "Kandy",
    "Nuwara Eliya": "Nuwara Eliya",
    "Galle": "Galle",
    "Mirissa": "Matara",
    "Ella": "Badulla",
    "Negombo": "Gampaha",
    "Polonnaruwa": "Polonnaruwa",
    "Sigiriya": "Matale",
    "Trincomalee": "Trincomalee",
    "Jaffna": "Jaffna",
    "Batticaloa": "Batticaloa",
    "Anuradhapura": "Anuradhapura",
    "Matara": "Matara",
    "Kalutara": "Kalutara",
    "Bentota": "Galle",
    "Hikkaduwa": "Galle",
    "Mirigama": "Gampaha",
    "Habarana": "Anuradhapura",
}

manual_location_mappings = {
    "Udawalawe National Park": {
        "province": "Sabaragamuwa Province",
        "district": "Ratnapura",
    },
    "North Central Province": {
        "province": "North Central Province",
        "district": "Anuradhapura",
    },
}

province_pattern = re.compile(
    r"(" + "|".join([re.escape(p) for p in provinces]) + r")", flags=re.IGNORECASE
)


def extract_province(location_str):
    if not isinstance(location_str, str) or not location_str.strip():
        return None

    loc = location_str.strip()

    for k, v in manual_location_mappings.items():
        if k.lower() == loc.lower():
            return v.get("province")

    m = province_pattern.search(loc)
    if m:
        matched = m.group(1)
        for p in provinces:
            if p.lower() == matched.lower():
                return p
        return matched

    m2 = re.search(r"([A-Za-z ]+Province)\\b", loc)
    if m2:
        return m2.group(1).strip()

    parts = [p.strip() for p in loc.split(",") if p.strip()]
    if parts:
        last = parts[-1]
        for p in provinces:
            if (
                last.lower() == p.replace(" Province", "").lower()
                or last.lower() == p.lower()
            ):
                return p

    return None


def infer_district(row):
    city = row.get("Located_City")
    location = row.get("Location")

    if isinstance(location, str):
        loc = location.strip()
        for k, v in manual_location_mappings.items():
            if k.lower() == loc.lower():
                return v.get("district")

    if isinstance(city, str) and city in city_to_district:
        return city_to_district[city]

    if not isinstance(location, str) or not location.strip():
        return None

    parts = [p.strip() for p in location.split(",") if p.strip()]

    for part in parts:
        if "District" in part:
            return part.replace("District", "").strip()

    if len(parts) == 1 and (
        any(parts[0].lower() == p.lower() for p in provinces)
        or "province" in parts[0].lower()
    ):
        if isinstance(city, str) and city.strip():
            if city in city_to_district:
                return city_to_district[city]
            return city
        return None

    if len(parts) >= 3:
        return parts[-2]

    if len(parts) == 2:
        first = parts[0]
        if city and isinstance(city, str) and city.lower() == first.lower():
            return city_to_district.get(city) or first
        return first

    if len(parts) == 1:
        single = parts[0]
        if (
            any(p.lower() == single.lower() for p in provinces)
            or "province" in single.lower()
        ):
            return None
        return single

    return None

In [40]:
# Validation-only cell: use the single canonical functions defined earlier (do not redefine them)
# This computes checks and lists remaining unmapped locations for review

# Compute check columns without redefining functions
df['province_check'] = df['Location'].apply(extract_province)
df['district_check'] = df.apply(infer_district, axis=1)

# Rows where either province or district could not be inferred
unmapped = df[df['province_check'].isnull() | df['district_check'].isnull()]

print("Unique Locations with missing province or district (examples):")
print(sorted(unmapped['Location'].dropna().unique())[:200])
print('\nCount of rows with missing province or district:', len(unmapped))

# show a few distinct examples for manual mapping decisions
display(unmapped[['Location','Located_City','province_check','district_check']].drop_duplicates().head(50))

Unique Locations with missing province or district (examples):
[]

Count of rows with missing province or district: 0


,Location,Located_City,province_check,district_check


## Location Feature Engineering (Province and District)

This cell adds `province` and `district` columns from `Location` and `Located_City`, then saves the enriched dataset to `processed_tourism_reviews_with_locations.csv`.

In [41]:
from pathlib import Path
import pandas as pd
import re

# Resolve paths for notebook execution context
project_dir = Path.cwd()
csv_in = project_dir / "Reviews.csv"
csv_out = project_dir / "processed_tourism_reviews_with_locations.csv"

df = pd.read_csv(csv_in, encoding="latin1")

# Known Sri Lankan provinces (standard names)
provinces = [
    "Western Province",
    "Central Province",
    "Southern Province",
    "Northern Province",
    "Eastern Province",
    "North Western Province",
    "North Central Province",
    "Uva Province",
    "Sabaragamuwa Province",
]

# Small city -> district mapping (best-effort for common locations seen in dataset)
city_to_district = {
    "Arugam Bay": "Ampara",
    "Colombo": "Colombo",
    "Kandy": "Kandy",
    "Nuwara Eliya": "Nuwara Eliya",
    "Galle": "Galle",
    "Mirissa": "Matara",
    "Ella": "Badulla",
    "Negombo": "Gampaha",
    "Polonnaruwa": "Polonnaruwa",
    "Sigiriya": "Matale",
    "Trincomalee": "Trincomalee",
    "Jaffna": "Jaffna",
    "Batticaloa": "Batticaloa",
    "Anuradhapura": "Anuradhapura",
    "Matara": "Matara",
    "Kalutara": "Kalutara",
    "Bentota": "Galle",
    "Hikkaduwa": "Galle",
    "Mirigama": "Gampaha",
    "Habarana": "Anuradhapura",
}

# Manual mappings for special Location strings
manual_location_mappings = {
    "Udawalawe National Park": {
        "province": "Sabaragamuwa Province",
        "district": "Ratnapura",
    },
    "North Central Province": {
        "province": "North Central Province",
        "district": "Anuradhapura",
    },
}

province_pattern = re.compile(
    r"(" + "|".join([re.escape(p) for p in provinces]) + r")", flags=re.IGNORECASE
)


def extract_province(location_str):
    if not isinstance(location_str, str) or not location_str.strip():
        return None

    loc = location_str.strip()

    # Manual mapping exact match (case-insensitive)
    for k, v in manual_location_mappings.items():
        if k.lower() == loc.lower():
            return v.get("province")

    # Direct match using known province names
    m = province_pattern.search(loc)
    if m:
        matched = m.group(1)
        for p in provinces:
            if p.lower() == matched.lower():
                return p
        return matched

    # Catch trailing "Province" text
    m2 = re.search(r"([A-Za-z ]+Province)\\b", loc)
    if m2:
        return m2.group(1).strip()

    # Comma-separated: last token might be province name without "Province"
    parts = [p.strip() for p in loc.split(",") if p.strip()]
    if parts:
        last = parts[-1]
        for p in provinces:
            if last.lower() == p.replace(" Province", "").lower() or last.lower() == p.lower():
                return p

    return None


def infer_district(row):
    city = row.get("Located_City")
    location = row.get("Location")

    # Manual mapping exact match
    if isinstance(location, str):
        loc = location.strip()
        for k, v in manual_location_mappings.items():
            if k.lower() == loc.lower():
                return v.get("district")

    # Prefer explicit city mapping
    if isinstance(city, str) and city in city_to_district:
        return city_to_district[city]

    if not isinstance(location, str) or not location.strip():
        return None

    parts = [p.strip() for p in location.split(",") if p.strip()]

    # If any part explicitly mentions "District", use it
    for part in parts:
        if "District" in part:
            return part.replace("District", "").strip()

    # If location is just a province name, try to use city as district
    if len(parts) == 1 and (
        any(parts[0].lower() == p.lower() for p in provinces)
        or "province" in parts[0].lower()
    ):
        if isinstance(city, str) and city.strip():
            if city in city_to_district:
                return city_to_district[city]
            return city
        return None

    # If parts are like "Town, District, Province" -> pick middle as district
    if len(parts) >= 3:
        return parts[-2]

    # If len==2 and first is likely a district/city, return first
    if len(parts) == 2:
        first = parts[0]
        if city and isinstance(city, str) and city.lower() == first.lower():
            return city_to_district.get(city) or first
        return first

    # Fallback: if the single token is not a province, return it
    if len(parts) == 1:
        single = parts[0]
        if any(p.lower() == single.lower() for p in provinces) or "province" in single.lower():
            return None
        return single

    return None


print("Extracting province and district (best-effort)...")

df["province"] = df["Location"].apply(extract_province)
df["district"] = df.apply(infer_district, axis=1)

# Normalize extracted text fields
for col in ["province", "district"]:
    df[col] = df[col].astype(object).apply(lambda x: x.strip() if isinstance(x, str) else x)

print("Province value counts (top 20):")
print(df["province"].value_counts(dropna=False).head(20))

print("\nSample districts (top 20):")
print(df["district"].value_counts(dropna=False).head(20))

print(f"Writing: {csv_out}")
df.to_csv(csv_out, index=False)
print("Done.")

Extracting province and district (best-effort)...
Province value counts (top 20):
province
Central Province          5252
North Central Province    3171
Southern Province         2648
Western Province          1890
Eastern Province          1162
Uva Province              1040
Sabaragamuwa Province      518
Northern Province          475
Name: count, dtype: int64

Sample districts (top 20):
district
Anuradhapura    2838
Kandy           2342
Nuwara Eliya    2221
Galle           1823
Colombo         1171
Matale           763
Jaffna           475
Badulla          471
Trincomalee      409
Haputale         409
Nilaveli         371
Pinnawala        369
Polonnaruwa      333
Beruwala         292
Ampara           222
Kalutara         218
Matara           217
Gampaha          209
Deniyaya         196
Weligatta        189
Name: count, dtype: int64
Writing: c:\Users\abaiy\Group-J---Research-Paper\processed_tourism_reviews_with_locations.csv
Done.
